# RL Deep Dives: Minari Library for Offline RL
## https://github.com/karthik-vi/MinariRL

## Presenters: Sai Karthik Varma Indukuri, Hemanth Poondla

Welcome to this code-based deep dive into **Minari**, the standard library for Offline Reinforcement Learning datasets, maintained by the Farama Foundation (the same team behind Gymnasium).

**What is Offline RL?**
Unlike standard online RL where the agent interacts with the environment, offline RL forces the agent to learn a policy *exclusively* from a fixed dataset of previously recorded interactions (transitions of state, action, reward, next state), without ever querying the live environment during training. Minari provides the standardized infrastructure to host, download, and create these datasets.

### Step 1: Installation & Setup
First, let's install Minari. The `[all]` tag ensures we get dependencies for common environments like D4RL (Deep Data-Driven RL).

In [ ]:
!pip install minari[all] gymnasium

### Step 2: Exploring Remote Datasets via Terminal Commands
Minari provides a powerful CLI (Command Line Interface). We can see what datasets are hosted on the remote servers.

In [ ]:
!minari list remote

### Step 3: Downloading Datasets
Let's download two datasets for the `door` environment from D4RL. One contains demonstrations from a human expert, and the other contains data from a trained RL expert.

In [ ]:
!minari download D4RL/door/human-v2
!minari download D4RL/door/expert-v2

In [ ]:
!minari list local

### Step 4: Loading and Inspecting Local Datasets
Now we transition to Python. We can load the dataset and inspect its metadata, which conforms to the Gymnasium API standards.

In [ ]:
import minari

dataset = minari.load_dataset('D4RL/door/human-v2')

print("Observation space:", dataset.observation_space)
print("Action space:", dataset.action_space)
print("Total episodes:", dataset.total_episodes)
print("Total steps:", dataset.total_steps)

### Step 5: Sampling Episodes
When training an offline RL agent (like Implicit Q-Learning or Decision Transformers), you need to sample trajectories from the dataset.

In [ ]:
dataset = minari.load_dataset("D4RL/door/human-v2")
dataset.set_seed(seed=123)

for i in range(5):
    # sample 5 episodes from the dataset
    episodes = dataset.sample_episodes(n_episodes=5)
    # get id's from the sampled episodes
    ids = list(map(lambda ep: ep.id, episodes))
    print(f"EPISODE ID'S SAMPLE {i}: {ids}")

### Step 6: Filtering the Dataset
Offline RL algorithms often perform better if you filter out bad trajectories. Minari allows you to apply lambda functions to filter episodes based on criteria like mean reward.

In [ ]:
dataset = minari.load_dataset("D4RL/door/human-v2")
print(f'TOTAL EPISODES ORIGINAL DATASET: {dataset.total_episodes}')

# Get episodes with mean reward greater than 2
filter_dataset = dataset.filter_episodes(lambda episode: episode.rewards.mean() > 2)
print(f'TOTAL EPISODES FILTER DATASET: {filter_dataset.total_episodes}')

### Step 7: Splitting Datasets
You can split datasets for Train/Validation/Test purposes.

In [ ]:
dataset = minari.load_dataset("D4RL/door/human-v2")
split_datasets = minari.split_dataset(dataset, sizes=[20, 5], seed=123)

print(f'TOTAL EPISODES FIRST SPLIT: {split_datasets[0].total_episodes}')
print(f'TOTAL EPISODES SECOND SPLIT: {split_datasets[1].total_episodes}')

### Step 8: Recovering the Original Environment
While offline RL agents don't interact with the environment during training, you still need the environment to *evaluate* them after they are trained. Minari stores the environment specifications directly in the dataset.

In [ ]:
dataset = minari.load_dataset('D4RL/door/human-v2')
env = dataset.recover_environment()

obs, info = env.reset()
print("Environment successfully recovered and reset.")
print("Initial Observation Shape:", obs.shape)

# Run a random policy to test it
for _ in range(100):
    obs, rew, terminated, truncated, info = env.step(env.action_space.sample())
    if terminated or truncated:
        env.reset()

### Step 9: Combining Datasets via CLI
You can combine the `human` and `expert` datasets into a single massive dataset right from the terminal.

In [ ]:
!minari combine D4RL/door/human-v2 D4RL/door/expert-v2 --dataset-id=D4RL/door/all-v0
!minari list local

### Step 10: Creating Your Own Dataset (The DataCollector)
What if you want to record your own custom dataset? Minari provides a `DataCollector` wrapper. Here, we will play `CartPole-v1` with a random policy and record 100 episodes to create a brand new dataset.

In [ ]:
from minari import DataCollector
import gymnasium as gym

# Wrap the environment
env = gym.make('CartPole-v1')
env = DataCollector(env, record_infos=True)

total_episodes = 100
for _ in range(total_episodes):
    env.reset(seed=123)
    while True:
        # random action policy
        action = env.action_space.sample()
        obs, rew, terminated, truncated, info = env.step(action)

        if terminated or truncated:
            break

# Save the collected data as a new Minari dataset
dataset = env.create_dataset(
    dataset_id="cartpole/test-v0",
    algorithm_name="Random-Policy",
    code_permalink="https://github.com/karthik-vi/MinariRL",
    author="Karthik Indukuri",
    author_email="sindukur@buffalo.edu"
)

print("Custom dataset successfully created!")
print("Total Episodes recorded:", dataset.total_episodes)


Minari standardizes how Offline RL datasets are created. By using the `DataCollector` wrapper and adhering to the Gymnasium API, it allows researchers to easily share, combine, filter, and load data, paving the way for the next generation of offline sequence modeling algorithms.


You can also upload your custom dataset on the web. How?
## Have Fun Exploring
### https://minari.farama.org/content/minari_cli/#upload-datasets